# Search-based inference: MCTS, DFS backtracking, and stepwise beam

A language model can propose next steps while a verifier or environment supplies a value. This notebook uses a tiny arithmetic state space so the mechanics are visible.

In [ ]:
from llm_textgen_lab.search import mcts

def expand(state):
    return [(state + 1, 0.6), (state + 2, 0.4)] if state < 8 else []

def evaluate(state):
    return 1.0 if state == 8 else -abs(8 - state) / 8

best = mcts(0, expand=expand, evaluate=evaluate, budget=50)
best.state, best.visits, best.mean_value

In [ ]:
from llm_textgen_lab.search import dfs_backtrack

# The preferred +2 branch overshoots past 7, so the search backs up and takes +3 from 4.
def propose(state):
    return [(state + 2, 0.6), (state + 3, 0.4)] if state < 7 else []

dfs_backtrack(0, expand=propose, is_goal=lambda state: state == 7, max_depth=5)

In [ ]:
from llm_textgen_lab.search import beam_steps

# Same proposer and value function as the MCTS cell; only the search changes.
beam_steps(0, expand=expand, score=evaluate, width=2, depth=4)

Contrast this with self-consistency: there, complete trajectories are sampled independently and never influence each other—voting is not search. Here the searches couple trajectories at intermediate states: branch, evaluate partial progress, and back up from dead ends. That branch-evaluate-backtrack loop is Tree of Thoughts, and modern reasoning models internalize the same move inside one long trace ("wait, let me reconsider"), which is why explicit tree search over free-form text is now mostly research-side.

For real text tasks, replace the arithmetic transition function with candidate continuations and replace `evaluate` with a test suite, exact checker, or trained verifier. Open-ended text is difficult because value is noisy. See [`docs/test-time-compute.md`](../docs/test-time-compute.md).